In [1]:
!pip install transformers datasets torch accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu

In [2]:
import torch
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import json
import re

2025-10-09 15:48:11.616051: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760024891.847514      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760024891.914228      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
model_name = "Qwen/Qwen3-8B"  
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Load model với quantization để tiết kiệm bộ nhớ
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    load_in_8bit=True
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

...

model-00002-of-00005.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.19G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
import json
from datetime import datetime
import torch
import re

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def load_data_from_json(path='/kaggle/input/question_1000.json'):
    """Load dữ liệu từ file JSON và chuẩn hóa sang cấu trúc internal dùng cho test_model."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            raw = json.load(f)
        print(f"Load dữ liệu từ JSON thành công: {path}")
    except Exception as e:
        print(f"Lỗi đọc file JSON: {e}")
        return None

    dataset = {
        'multiple_choice': [],
        'true_false': []
    }

    mc_raw = raw.get('multiple_choice', [])
    for idx, item in enumerate(mc_raw):
        q_text = item.get('question') or item.get('title') or ""
        raw_options = item.get('options', [])

        # Nếu options là list of strings (tình huống khác), xử lý tương tự
        normalized_opts = []
        correct_letter = None
        for i, opt in enumerate(raw_options):
            if isinstance(opt, dict):
                ans_text = opt.get('answer', '').strip()
                is_correct = bool(opt.get('isCorrect', False))
            else:
                # opt có thể là string
                ans_text = str(opt).strip()
                is_correct = False

            letter = LETTERS[i] if i < len(LETTERS) else f"Opt{i+1}"
            normalized_opts.append(f"{letter}. {ans_text}")
            if is_correct and correct_letter is None:
                correct_letter = letter

        if correct_letter is None:
            for k in ('correct_answer', 'answer', 'correct'):
                if k in item:
                    val = str(item[k]).strip().upper()
                    # nếu val là 'A' hoặc '1'...
                    if len(val) == 1 and val in LETTERS:
                        correct_letter = val
                        break
                    # nếu val chứa text tương ứng, tìm match
                    for i, opt in enumerate(raw_options):
                        opt_text = opt.get('answer','').strip() if isinstance(opt, dict) else str(opt).strip()
                        if opt_text and val in opt_text.upper():
                            correct_letter = LETTERS[i]
                            break
                    if correct_letter:
                        break

        # Nếu vẫn None -> đặt mặc định là 'A' (không nên nhưng tránh crash)
        if correct_letter is None and normalized_opts:
            correct_letter = LETTERS[0]

        dataset['multiple_choice'].append({
            'question': q_text,
            'options': normalized_opts,            # list of "A. text" strings
            'correct_answer': correct_letter
        })

    tf_raw = raw.get('true_false', [])
    for idx, item in enumerate(tf_raw):
        q_text = item.get('question') or item.get('text') or ""
        raw_options = item.get('options', [])

        correct_tf = None
        for opt in raw_options:
            if isinstance(opt, dict):
                ans_text = opt.get('answer','').strip().upper()
                is_correct = bool(opt.get('isCorrect', False))
                if is_correct:
                    if 'ĐÚNG' in ans_text or 'DUNG' in ans_text or 'TRUE' in ans_text:
                        correct_tf = 'T'
                    elif 'SAI' in ans_text or 'FALSE' in ans_text:
                        correct_tf = 'F'
        # Nếu dataset có trường chỉ định trực tiếp
        if correct_tf is None:
            for k in ('correct_answer', 'answer', 'correct'):
                if k in item:
                    val = str(item[k]).strip().upper()
                    if val in ('ĐÚNG','DUNG','TRUE','T','YES'):
                        correct_tf = 'T'
                        break
                    if val in ('SAI','FALSE','F','NO'):
                        correct_tf = 'F'
                        break
                    if len(val) == 1 and val in ('T','F','A','B'):
                        # nếu người ta để 'A'/'B' tương ứng, thử map theo option text
                        if raw_options and isinstance(raw_options, list):
                            # map A->first option etc
                            # not robust but best-effort
                            if val == 'A':
                                opt0 = raw_options[0]
                                txt = opt0.get('answer','').upper() if isinstance(opt0, dict) else str(opt0).upper()
                                correct_tf = 'T' if ('ĐÚNG' in txt or 'TRUE' in txt) else 'F'
                                break

        if correct_tf is None and raw_options:
            for opt in raw_options:
                txt = opt.get('answer','').upper() if isinstance(opt, dict) else str(opt).upper()
                if 'ĐÚNG' in txt or 'DUNG' in txt or 'TRUE' in txt:
                    correct_tf = 'T' if ('ĐÚNG' in txt or 'TRUE' in txt) else 'F'
                    break

        # Nếu vẫn không biết -> đặt 'T' mặc định (để tránh lỗi)
        if correct_tf is None:
            correct_tf = 'T'

        # Chuẩn hóa thành structure mà test_model dùng: main_question, sub_questions (một câu), correct_answers list
        dataset['true_false'].append({
            'main_question': q_text,
            'sub_questions': [
                {'letter': 'A', 'text': q_text}  # single statement as one sub-question
            ],
            'correct_answers': [correct_tf]
        })

    return dataset


def extract_answer(response, is_tf=False, num_tf=0):
    """Trích xuất đáp án từ response"""
    if is_tf:
        # Extract T/F answers
        answers = []
        patterns = [
            r'\(([A-D])\)\s*:?\s*(ĐÚNG|SAI|TRUE|FALSE|T|F)',
            r'([A-D])\.\s*:?\s*(ĐÚNG|SAI|TRUE|FALSE|T|F)',
            r'\b(ĐÚNG|SAI|TRUE|FALSE|T|F)\b',
            r'\b(T|F)\b'
        ]

        response_upper = response.upper()
        for pattern in patterns:
            matches = re.findall(pattern, response_upper)
            for match in matches:
                if isinstance(match, tuple):
                    tf_part = match[-1]
                else:
                    tf_part = match
                if tf_part in ['ĐÚNG', 'TRUE', 'T', 'DUNG', 'ĐUNG']:
                    answers.append('T')
                elif tf_part in ['SAI', 'FALSE', 'F']:
                    answers.append('F')

                if len(answers) >= num_tf:
                    break
            if len(answers) >= num_tf:
                break

        # Pad with T if not enough answers
        while len(answers) < num_tf:
            answers.append('T')

        return answers[:num_tf]
    else:
        # Extract A/B/C/D answer
        patterns = [
            r'ĐÁP ÁN ĐÚNG LÀ\s*\(?([A-D])\)?',
            r'ĐÁP ÁN\s*:?\s*\(?([A-D])\)?',
            r'CHỌN\s*\(?([A-D])\)?',
            r'\(([A-D])\)',
            r'\b([A-D])\b'
        ]

        response_upper = response.upper()
        for pattern in patterns:
            match = re.search(pattern, response_upper)
            if match:
                return match.group(1)

        return None


def test_model(questions_list, question_type="MC"):
    """Test model với câu hỏi"""
    print(f"\nĐÁNH GIÁ MODEL - {question_type}")
    print("=" * 50)

    correct_count = 0
    total_count = 0

    for i, item in enumerate(questions_list, 1):
        print(f"Câu {i}/{len(questions_list)}...", end=" ")

        try:
            if question_type == "MC":
                # Multiple choice prompt
                options_text = "\n".join(item['options'])
                prompt = f"""Bạn là chuyên gia lịch sử, am hiểu về kiến thức lịch sử Việt Nam và thế giới. Trả lời câu hỏi:

{item['question']}
{options_text}

Đáp án:"""

                # Generate với các tham số cố định
                inputs = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
                with torch.no_grad():
                    outputs = model.generate(
                        inputs,
                        max_new_tokens=30,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id,
                        eos_token_id=tokenizer.eos_token_id
                    )

                response = tokenizer.decode(outputs[0], skip_special_tokens=True)
                model_response = response[len(prompt):].strip()

                predicted = extract_answer(model_response)
                correct = item['correct_answer']

                if predicted == correct:
                    correct_count += 1
                    print("ĐÚNG")
                else:
                    print(f"SAI (pred={predicted} expected={correct})")
                total_count += 1

            else:  # TF
                # True/False prompt
                sub_text = ""
                for sub in item['sub_questions']:
                    sub_text += f"({sub['letter']}) {sub['text']}\n"

                prompt = f"""Bạn là một chuyên gia lịch sử, am hiểu lịch sử Việt Nam và thế giới. Đánh giá đúng/sai các câu sau. Trả lời T (đúng) hoặc F (sai):

{item['main_question']}

{sub_text}

Trả lời:"""

                inputs = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
                with torch.no_grad():
                    outputs = model.generate(
                        inputs,
                        max_new_tokens=50,
                        do_sample=False,
                        pad_token_id=tokenizer.eos_token_id,
                        eos_token_id=tokenizer.eos_token_id
                    )

                response = tokenizer.decode(outputs[0], skip_special_tokens=True)
                model_response = response[len(prompt):].strip()

                num_sub = len(item['sub_questions'])
                predicted = extract_answer(model_response, is_tf=True, num_tf=num_sub)
                correct = item['correct_answers']

                # Count correct sub-answers
                sub_correct = 0
                min_len = min(len(predicted), len(correct), num_sub)
                for j in range(min_len):
                    if predicted[j] == correct[j]:
                        sub_correct += 1
                        correct_count += 1
                    total_count += 1

                # Add remaining questions to total
                total_count += max(0, num_sub - min_len)

                print(f"ĐÚNG {sub_correct}/{min_len}" if sub_correct == min_len else f"SAI ({sub_correct}/{min_len})")

        except Exception as e:
            print(f"Lỗi: {e}")
            if question_type == "MC":
                total_count += 1
            else:
                total_count += len(item.get('sub_questions', []))

    accuracy = (correct_count / total_count) * 100 if total_count > 0 else 0
    print(f"\nKẾT QUẢ {question_type}:")
    print(f"Đúng: {correct_count}/{total_count} = {accuracy:.2f}%")

    return accuracy


def main_evaluation(json_path='/kaggle/input/question_1000.json'):
    """Đánh giá chính"""
    print("BẮT ĐẦU ĐÁNH GIÁ")
    print("=" * 50)
    print(f"Thời gian: {datetime.now().strftime('%H:%M:%S')}")

    # Load data from JSON (and normalize)
    dataset = load_data_from_json(json_path)

    if not dataset:
        print("Thiếu dữ liệu!")
        return

    # Test và tính độ chính xác
    mc_accuracy = 0
    tf_accuracy = 0

    try:
        mc_questions = dataset.get('multiple_choice', [])
        if mc_questions:
            print(f"Tìm thấy {len(mc_questions)} câu hỏi trắc nghiệm")
            mc_accuracy = test_model(mc_questions, "MC")
        else:
            print("Không có câu hỏi trắc nghiệm trong dataset")
    except Exception as e:
        print(f"Lỗi xử lý trắc nghiệm: {e}")

    try:
        tf_questions = dataset.get('true_false', [])
        if tf_questions:
            print(f"Tìm thấy {len(tf_questions)} câu hỏi đúng/sai")
            tf_accuracy = test_model(tf_questions, "TF")
        else:
            print("Không có câu hỏi đúng/sai trong dataset")
    except Exception as e:
        print(f"Lỗi xử lý đúng/sai: {e}")

    # Tổng kết
    print("\n" + "=" * 50)
    print("TỔNG KẾT")
    print("=" * 50)
    print(f"Trắc nghiệm: {mc_accuracy:.2f}%")
    print(f"Đúng/Sai: {tf_accuracy:.2f}%")
    if mc_accuracy > 0 and tf_accuracy > 0:
        overall_accuracy = (mc_accuracy + tf_accuracy) / 2
        print(f"Tổng thể: {overall_accuracy:.2f}%")
    print(f"Hoàn thành: {datetime.now().strftime('%H:%M:%S')}")


# Chạy đánh giá
if __name__ == "__main__":
    try:
        main_evaluation('/kaggle/input/question_1000.json')
    except Exception as e:
        print(f"Lỗi tổng thể: {e}")
        import traceback
        traceback.print_exc()


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


BẮT ĐẦU ĐÁNH GIÁ
Thời gian: 15:51:34
Load dữ liệu từ JSON thành công: /kaggle/input/question_1000.json
Tìm thấy 357 câu hỏi trắc nghiệm

ĐÁNH GIÁ MODEL - MC
Câu 1/357... 

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ĐÚNG
Câu 2/357... 

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

ĐÚNG 1/1
Câu 722/724... 

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SAI (0/1)
Câu 723/724... 

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SAI (0/1)
Câu 724/724... ĐÚNG 1/1

KẾT QUẢ TF:
Đúng: 390/724 = 53.87%

TỔNG KẾT
Trắc nghiệm: 56.30%
Đúng/Sai: 53.87%
Tổng thể: 55.08%
Hoàn thành: 18:49:48
